# 01_setup_data.ipynb

Prepare the Etsy challenge datasets for the rest of the pipeline.

## What this notebook does
- locates the project root robustly
- defines and creates the project directories
- optionally extracts archives
- validates the Etsy `train.csv`, `test.csv`, and image folders
- optionally ingests external labelled image sources
- builds clean manifest files for downstream notebooks

## Main outputs
- `outputs/manifests/train_manifest_etsy_only.csv`
- `outputs/manifests/test_manifest_etsy.csv`
- `outputs/manifests/train_manifest_etsy_plus_external.csv`
- `outputs/manifests/dataset_summary.csv`

## Notes
- This notebook does **not** train any model.
- It keeps the original Etsy CSV files untouched.
- External data is added via derived manifests, not by overwriting the original training set.


## 1. Imports

In [9]:
from __future__ import annotations

from pathlib import Path
from zipfile import ZipFile
import json
import os
import random
import shutil
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 2. Configuration

Edit only this cell when you want to:
- change the detected project root
- enable archive extraction
- register external datasets
- choose whether to save image metadata


In [75]:
CONFIG = {
    # --- project root detection ---
    "project_root_override": None,
    "root_markers": ["data2", "notebooks"],

    # --- Etsy challenge ---
    "challenge_rel_dir": "data2/etsy_challenge",
    "train_csv_name": "train.csv",
    "test_csv_name": "test.csv",

    # Shared image folder used by both train.csv and test.csv
    "shared_image_dirs_candidates": [
        "images_final_sample",
        "final_test_images",
        "images",
        "train_images"
    ],

    # --- archive extraction ---
    "extract_archives": True,
    "archive_specs": [
        {
            "zip_path": "data2/etsy_challenge/genai_image_challenge.zip",
            "extract_to": "data2/etsy_challenge"
        }
    ],
    "overwrite_extracted_folders": False,

    # --- outputs ---
    "outputs_rel_dir": "outputs",
    "save_image_metadata": True,
    "sample_size_for_quick_checks": None,

    # --- label handling ---
    # External datasets will use these canonical string labels.
    "positive_label": "ai_generated",
    "negative_label": "authentic",

    # Etsy numeric label mapping
    "etsy_label_map": {
        1: "ai_generated",
        0: "authentic"
    },

    # --- external datasets ---
    "external_sources": [
        {
            "name": "ai5_3",
            "enabled": True,
            "type": "folder_by_class",
            "root_dir": "data2/other_datasets/kaggle_datasets/ai5_3",
            "authentic_subdir": "real_dataset",
            "ai_generated_subdir": "Ai_generated_dataset",
            "glob_patterns": ["*.jpg", "*.jpeg", "*.png", "*.webp"]
        },
        {
            "name": "scraped_2",
            "enabled": False,
            "type": "folder_by_class",
            "root_dir": "data2/other_datasets/kaggle_datasets/scraped_2",
            "authentic_subdir": "RealArt",
            "ai_generated_subdir": "AiArtData",
            "glob_patterns": ["*.jpg", "*.jpeg", "*.png", "*.webp"]
        },
        {
            "name": "shoes_1",
            "enabled": False,
            "type": "folder_by_class",
            "root_dir": "data2/other_datasets/kaggle_datasets/shoes_1",
            "authentic_subdir": "real",
            "ai_generated_subdir": "ai-midjourney",
            "glob_patterns": ["*.jpg", "*.jpeg", "*.png", "*.webp"]
        }
    ],
}

## 3. Helper functions

In [62]:
def _resolve_path(project_root: Path, p: str | Path) -> Path:
    p = Path(p)
    return p if p.is_absolute() else project_root / p


def find_project_root(start: Path, markers: list[str]) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        score = sum((candidate / marker).exists() for marker in markers)
        if score >= 1:
            return candidate
    raise FileNotFoundError(
        f"Could not detect project root from {start}. "
        f"Tried markers: {markers}"
    )


def choose_existing_dir(base_dir: Path, candidates: list[str]) -> Optional[Path]:
    for name in candidates:
        p = base_dir / name
        if p.exists() and p.is_dir():
            return p
    return None


def extract_zip_if_needed(zip_path: Path, extract_to: Path, overwrite: bool = False) -> None:
    if not zip_path.exists():
        raise FileNotFoundError(f"Archive not found: {zip_path}")

    extract_to.mkdir(parents=True, exist_ok=True)

    # Consider the folder "already extracted" only if it contains something
    # other than the archive file itself.
    existing_items = [p for p in extract_to.iterdir() if p.resolve() != zip_path.resolve()]

    if not overwrite and existing_items:
        print(f"[SKIP] Extraction target already contains extracted content: {extract_to}")
        return

    print(f"[EXTRACT] {zip_path} -> {extract_to}")
    with ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)

def is_valid_image(path: Path) -> bool:
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except (UnidentifiedImageError, OSError, FileNotFoundError):
        return False


def collect_images_recursive(root_dir: Path, glob_patterns: list[str]) -> list[Path]:
    files = []
    for pattern in glob_patterns:
        files.extend(root_dir.rglob(pattern))
    # deterministic ordering + dedupe
    files = sorted(set(files))
    return files


def add_basic_image_metadata(df: pd.DataFrame, image_path_col: str = "image_path") -> pd.DataFrame:
    widths, heights, modes, formats = [], [], [], []
    for p in df[image_path_col]:
        try:
            with Image.open(p) as img:
                widths.append(img.width)
                heights.append(img.height)
                modes.append(img.mode)
                formats.append(img.format)
        except Exception:
            widths.append(None)
            heights.append(None)
            modes.append(None)
            formats.append(None)
    out = df.copy()
    out["width"] = widths
    out["height"] = heights
    out["mode"] = modes
    out["format"] = formats
    out["aspect_ratio"] = out["width"] / out["height"]
    return out


def normalise_label(value: str, positive_label: str, negative_label: str) -> str:
    value = str(value).strip().lower()
    mapping = {
        positive_label.lower(): positive_label,
        negative_label.lower(): negative_label,
        "ai": positive_label,
        "ai_generated": positive_label,
        "generated": positive_label,
        "fake": positive_label,
        "synthetic": positive_label,
        "real": negative_label,
        "authentic": negative_label,
        "non_ai": negative_label,
        "human": negative_label,
    }
    if value not in mapping:
        raise ValueError(f"Unsupported label value: {value}")
    return mapping[value]


def build_etsy_train_manifest(
    train_df: pd.DataFrame,
    image_dir: Path,
    etsy_label_map: dict,
    sample_size: int | None = None,
) -> pd.DataFrame:

    df = train_df.copy()

    if sample_size:
        df = df.sample(n=sample_size, random_state=42)

    df["image_path"] = df["image_id"].apply(lambda x: image_dir / str(x))

    df["label"] = df["ground_truth"].map(etsy_label_map)

    df["source"] = "etsy"
    df["split"] = "train"

    df["file_exists"] = df["image_path"].apply(Path.exists)
    df["valid_image"] = df["image_path"].apply(is_valid_image)

    return df[[
        "image_id",
        "image_path",
        "label",
        "source",
        "split",
        "file_exists",
        "valid_image",
    ]]


def build_etsy_test_manifest(
    test_df: pd.DataFrame,
    image_dir: Path,
    sample_size: int | None = None,
) -> pd.DataFrame:

    df = test_df.copy()

    if sample_size:
        df = df.sample(n=sample_size, random_state=42)

    df["image_path"] = df["image_id"].apply(lambda x: image_dir / str(x))

    df["label"] = pd.NA
    df["source"] = "etsy"
    df["split"] = "test"

    df["file_exists"] = df["image_path"].apply(Path.exists)
    df["valid_image"] = df["image_path"].apply(is_valid_image)

    return df[[
        "image_id",
        "image_path",
        "label",
        "source",
        "split",
        "file_exists",
        "valid_image",
    ]]


def build_external_manifest(project_root: Path, source_cfg: dict, positive_label: str, negative_label: str) -> pd.DataFrame:
    source_type = source_cfg["type"]
    source_name = source_cfg["name"]

    if source_type == "folder_by_class":
        root_dir = _resolve_path(project_root, source_cfg["root_dir"])
        authentic_dir = root_dir / source_cfg.get("authentic_subdir", negative_label)
        ai_dir = root_dir / source_cfg.get("ai_generated_subdir", positive_label)
        glob_patterns = source_cfg.get("glob_patterns", ["*.jpg", "*.jpeg", "*.png", "*.webp"])

        rows = []
        for label, class_dir in [(negative_label, authentic_dir), (positive_label, ai_dir)]:
            if not class_dir.exists():
                print(f"[WARN] Missing external class directory for {source_name}: {class_dir}")
                continue
            for image_path in collect_images_recursive(class_dir, glob_patterns):
                rows.append({
                    "image_id": f"{source_name}/{image_path.relative_to(root_dir).as_posix()}",
                    "image_path": image_path,
                    "label": label,
                    "source": source_name,
                    "split": "train",
                    "file_exists": image_path.exists(),
                    "valid_image": is_valid_image(image_path),
                })

        return pd.DataFrame(rows)

    elif source_type == "csv_with_paths":
        csv_path = _resolve_path(project_root, source_cfg["csv_path"])
        image_path_col = source_cfg.get("image_path_col", "image_path")
        label_col = source_cfg.get("label_col", "label")

        ext_df = pd.read_csv(csv_path)
        required = {image_path_col, label_col}
        missing = required - set(ext_df.columns)
        if missing:
            raise ValueError(f"External source {source_name} missing columns: {missing}")

        out = pd.DataFrame({
            "image_id": ext_df[image_path_col].apply(lambda x: Path(str(x)).name),
            "image_path": ext_df[image_path_col].apply(lambda x: _resolve_path(project_root, str(x))),
            "label": ext_df[label_col].apply(lambda x: normalise_label(x, positive_label=positive_label, negative_label=negative_label)),
            "source": source_name,
            "split": "train",
        })
        out["file_exists"] = out["image_path"].apply(Path.exists)
        out["valid_image"] = out["image_path"].apply(is_valid_image)
        return out

    else:
        raise ValueError(f"Unsupported external source type: {source_type}")

def find_shared_image_dir(challenge_dir: Path, candidates: list[str]) -> Path:
    for rel in candidates:
        candidate = challenge_dir / rel
        if candidate.exists() and candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find a shared image directory under {challenge_dir}. "
        f"Tried: {candidates}"
    )


## 4. Locate project root and define paths

In [76]:
cwd = Path.cwd()

if CONFIG["project_root_override"]:
    PROJECT_ROOT = Path(CONFIG["project_root_override"]).expanduser().resolve()
else:
    PROJECT_ROOT = find_project_root(cwd, CONFIG["root_markers"])

CHALLENGE_DIR = PROJECT_ROOT / CONFIG["challenge_rel_dir"]

IMAGE_DIR = find_shared_image_dir(
    CHALLENGE_DIR,
    CONFIG["shared_image_dirs_candidates"]
)

TRAIN_CSV = CHALLENGE_DIR / CONFIG["train_csv_name"]
TEST_CSV = CHALLENGE_DIR / CONFIG["test_csv_name"]

OUTPUTS_DIR = _resolve_path(PROJECT_ROOT, CONFIG["outputs_rel_dir"])
MANIFESTS_DIR = OUTPUTS_DIR / "manifests"
REPORTS_DIR = OUTPUTS_DIR / "reports"
TMP_DIR = OUTPUTS_DIR / "tmp"

for d in [OUTPUTS_DIR, MANIFESTS_DIR, REPORTS_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("CHALLENGE_DIR:", CHALLENGE_DIR)
print("TRAIN_CSV    :", TRAIN_CSV)
print("TEST_CSV     :", TEST_CSV)
print("IMAGE_DIR    :", IMAGE_DIR)
print("OUTPUTS_DIR  :", OUTPUTS_DIR)

PROJECT_ROOT : /home/tbeliere/Projects/DCU/AML_PJ
CHALLENGE_DIR: /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge
TRAIN_CSV    : /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge/train.csv
TEST_CSV     : /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge/test.csv
IMAGE_DIR    : /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge/images_final_sample
OUTPUTS_DIR  : /home/tbeliere/Projects/DCU/AML_PJ/outputs


## 5. Optional archive extraction

Use this only if you still need to extract one or more zip archives.
If your data is already extracted correctly, this step will simply do nothing.


In [64]:
if CONFIG["extract_archives"] and CONFIG["archive_specs"]:
    for spec in CONFIG["archive_specs"]:
        zip_path = _resolve_path(PROJECT_ROOT, spec["zip_path"])
        extract_to = _resolve_path(PROJECT_ROOT, spec["extract_to"])
        extract_zip_if_needed(
            zip_path=zip_path,
            extract_to=extract_to,
            overwrite=CONFIG["overwrite_extracted_folders"]
        )
else:
    print("No archive extraction requested.")


[SKIP] Extraction target already contains extracted content: /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge


In [65]:
import pandas as pd

train_df = pd.read_csv(PROJECT_ROOT / "data2" / "etsy_challenge" / "train.csv")
test_df = pd.read_csv(PROJECT_ROOT / "data2" / "etsy_challenge" / "test.csv")

print("TRAIN COLUMNS:", train_df.columns.tolist())
print("TEST COLUMNS:", test_df.columns.tolist())

print("\nTRAIN SAMPLE:")
print(train_df.head())

print("\nTEST SAMPLE:")
print(test_df.head())

TRAIN COLUMNS: ['image_id', 'ground_truth']
TEST COLUMNS: ['image_id']

TRAIN SAMPLE:
                                   image_id  ground_truth
0  e7ea0752-f24a-4dac-926c-371fda631a0f.jpg             1
1  cdc7b8e5-1609-4f88-a20d-0dd321f8f489.jpg             0
2  c954e18f-8ab3-4bff-91d7-a0d7ff000e51.jpg             0
3  1c5fb5d5-75e0-431f-a03e-e50026f23fe3.jpg             0
4  fd0dd104-7300-4da3-a46f-6980531ead33.jpg             1

TEST SAMPLE:
                                   image_id
0  3ecf1af5-6a8f-416a-9b4c-df9f2e0a0a80.jpg
1  2789b3fe-a337-4dc2-b42c-8bccde1f68fb.jpg
2  01a342c6-c3fc-4b55-8c22-13c1a556ba87.jpg
3  ac784910-b461-498d-b3a8-50b1e4116b11.jpg
4  6dcd4df6-7447-4bcf-a29b-f7f53b4c3ed4.jpg


## 6. Sanity checks for the Etsy challenge dataset

In [71]:
from IPython.display import display

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Missing Etsy train.csv: {TRAIN_CSV}")

if not TEST_CSV.exists():
    raise FileNotFoundError(f"Missing Etsy test.csv: {TEST_CSV}")

if IMAGE_DIR is None:
    raise FileNotFoundError(
        f"Could not find the Etsy shared image directory inside {CHALLENGE_DIR}. "
        f"Tried: {CONFIG['shared_image_dirs_candidates']}"
    )

train_df_raw = pd.read_csv(TRAIN_CSV)
test_df_raw = pd.read_csv(TEST_CSV)

print("train.csv shape:", train_df_raw.shape)
print("test.csv shape :", test_df_raw.shape)
print("IMAGE_DIR      :", IMAGE_DIR)

display(train_df_raw.head())
display(test_df_raw.head())

train.csv shape: (4800, 2)
test.csv shape : (2058, 1)
IMAGE_DIR      : /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge/images_final_sample


,image_id,ground_truth
0,e7ea0752-f24a-4dac-926c-371fda631a0f.jpg,1
1,cdc7b8e5-1609-4f88-a20d-0dd321f8f489.jpg,0
2,c954e18f-8ab3-4bff-91d7-a0d7ff000e51.jpg,0
3,1c5fb5d5-75e0-431f-a03e-e50026f23fe3.jpg,0
4,fd0dd104-7300-4da3-a46f-6980531ead33.jpg,1


,image_id
0,3ecf1af5-6a8f-416a-9b4c-df9f2e0a0a80.jpg
1,2789b3fe-a337-4dc2-b42c-8bccde1f68fb.jpg
2,01a342c6-c3fc-4b55-8c22-13c1a556ba87.jpg
3,ac784910-b461-498d-b3a8-50b1e4116b11.jpg
4,6dcd4df6-7447-4bcf-a29b-f7f53b4c3ed4.jpg


## 7. Build Etsy manifests

In [72]:
# --- build Etsy manifests using shared image directory ---

train_manifest_etsy = build_etsy_train_manifest(
    train_df=train_df_raw,
    image_dir=IMAGE_DIR,
    etsy_label_map=CONFIG["etsy_label_map"],
    sample_size=CONFIG["sample_size_for_quick_checks"],
)

test_manifest_etsy = build_etsy_test_manifest(
    test_df=test_df_raw,
    image_dir=IMAGE_DIR,
    sample_size=CONFIG["sample_size_for_quick_checks"],
)

# --- optional metadata enrichment ---
if CONFIG["save_image_metadata"]:
    train_manifest_etsy = add_basic_image_metadata(train_manifest_etsy)
    test_manifest_etsy = add_basic_image_metadata(test_manifest_etsy)

# --- preview ---
display(train_manifest_etsy.head())
display(test_manifest_etsy.head())


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio
0,e7ea0752-f24a-4dac-926c-371fda631a0f.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000
1,cdc7b8e5-1609-4f88-a20d-0dd321f8f489.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,authentic,etsy,train,True,True,2404,2404,RGB,JPEG,1.000000
2,c954e18f-8ab3-4bff-91d7-a0d7ff000e51.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,authentic,etsy,train,True,True,1195,896,RGB,JPEG,1.333705
3,1c5fb5d5-75e0-431f-a03e-e50026f23fe3.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,authentic,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000
4,fd0dd104-7300-4da3-a46f-6980531ead33.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,716,RGB,JPEG,1.430168


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio
0,3ecf1af5-6a8f-416a-9b4c-df9f2e0a0a80.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,<NA>,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
1,2789b3fe-a337-4dc2-b42c-8bccde1f68fb.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,<NA>,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
2,01a342c6-c3fc-4b55-8c22-13c1a556ba87.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,<NA>,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
3,ac784910-b461-498d-b3a8-50b1e4116b11.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,<NA>,etsy,test,True,True,400,300,RGB,JPEG,1.333333
4,6dcd4df6-7447-4bcf-a29b-f7f53b4c3ed4.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,<NA>,etsy,test,True,True,832,1248,RGB,JPEG,0.666667


## 8. Validate Etsy manifests

In [73]:
def summarise_manifest(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = {
        "manifest_name": name,
        "n_rows": len(df),
    }

    # --- file checks ---
    if "file_exists" in df.columns:
        summary["n_missing_files"] = int((~df["file_exists"]).sum())
    else:
        summary["n_missing_files"] = None

    if "valid_image" in df.columns:
        summary["n_invalid_images"] = int((~df["valid_image"]).sum())
    else:
        summary["n_invalid_images"] = None

    # --- label distribution ---
    if "label" in df.columns:
        label_counts = df["label"].value_counts(dropna=False).to_dict()
        for k, v in label_counts.items():
            summary[f"label__{k}"] = int(v)

    # --- source distribution ---
    if "source" in df.columns:
        source_counts = df["source"].value_counts(dropna=False).to_dict()
        for k, v in source_counts.items():
            summary[f"source__{k}"] = int(v)

    return pd.DataFrame([summary])


def print_manifest_warnings(df: pd.DataFrame, name: str):
    print(f"\n--- Checking {name} ---")

    if "file_exists" in df.columns:
        n_missing = (~df["file_exists"]).sum()
        if n_missing > 0:
            print(f"[WARN] {name}: {n_missing} missing image files")

    if "valid_image" in df.columns:
        n_invalid = (~df["valid_image"]).sum()
        if n_invalid > 0:
            print(f"[WARN] {name}: {n_invalid} unreadable/corrupted images")


# --- summaries ---
etsy_train_summary = summarise_manifest(train_manifest_etsy, "train_manifest_etsy_only")
etsy_test_summary = summarise_manifest(test_manifest_etsy, "test_manifest_etsy")

display(etsy_train_summary)
display(etsy_test_summary)

# --- warnings ---
print_manifest_warnings(train_manifest_etsy, "train_manifest_etsy_only")
print_manifest_warnings(test_manifest_etsy, "test_manifest_etsy")

,manifest_name,n_rows,n_missing_files,n_invalid_images,label__authentic,label__ai_generated,source__etsy
0,train_manifest_etsy_only,4800,0,0,2485,2315,4800


,manifest_name,n_rows,n_missing_files,n_invalid_images,label__<NA>,source__etsy
0,test_manifest_etsy,2058,0,0,2058,2058



--- Checking train_manifest_etsy_only ---

--- Checking test_manifest_etsy ---


In [57]:
print("Missing train:", (~train_manifest_etsy["file_exists"]).sum())
print("Missing test :", (~test_manifest_etsy["file_exists"]).sum())

Missing train: 0
Missing test : 0


## 9. Ingest optional external labelled datasets

Enable one or more `external_sources` in the config cell if you want to build a second training manifest that includes external data.


In [77]:
external_frames = []

enabled_sources = [src for src in CONFIG["external_sources"] if src.get("enabled", False)]
print("Enabled external sources:", [src["name"] for src in enabled_sources])

for src in enabled_sources:
    print(f"\n[LOAD EXTERNAL] {src['name']}")
    ext_df = build_external_manifest(
        project_root=PROJECT_ROOT,
        source_cfg=src,
        positive_label=CONFIG["positive_label"],
        negative_label=CONFIG["negative_label"],
    )

    if ext_df.empty:
        print(f"[WARN] External source produced no rows: {src['name']}")
        continue

    if CONFIG["save_image_metadata"]:
        ext_df = add_basic_image_metadata(ext_df)

    display(ext_df.head())
    external_frames.append(ext_df)


Enabled external sources: ['ai5_3']

[LOAD EXTERNAL] ai5_3


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio
0,ai5_3/real_dataset/animals/animals_0.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,authentic,ai5_3,train,True,True,1080,1440,RGB,JPEG,0.750000
1,ai5_3/real_dataset/animals/animals_1.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,authentic,ai5_3,train,True,True,1080,720,RGB,JPEG,1.500000
2,ai5_3/real_dataset/animals/animals_10.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,authentic,ai5_3,train,True,True,1080,720,RGB,JPEG,1.500000
3,ai5_3/real_dataset/animals/animals_100.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,authentic,ai5_3,train,True,True,1080,694,RGB,JPEG,1.556196
4,ai5_3/real_dataset/animals/animals_101.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,authentic,ai5_3,train,True,True,1080,1259,RGB,JPEG,0.857824


## 10. Create final training manifests

In [78]:
train_manifest_etsy_only = train_manifest_etsy.copy()

if external_frames:
    external_manifest_all = pd.concat(external_frames, ignore_index=True)
    train_manifest_etsy_plus_external = pd.concat(
        [train_manifest_etsy_only, external_manifest_all],
        ignore_index=True
    )
else:
    external_manifest_all = pd.DataFrame(columns=train_manifest_etsy_only.columns)
    train_manifest_etsy_plus_external = train_manifest_etsy_only.copy()

# Keep deterministic order
sort_cols = [c for c in ["source", "label", "image_id"] if c in train_manifest_etsy_plus_external.columns]
if sort_cols:
    train_manifest_etsy_only = train_manifest_etsy_only.sort_values(sort_cols).reset_index(drop=True)
    train_manifest_etsy_plus_external = train_manifest_etsy_plus_external.sort_values(sort_cols).reset_index(drop=True)
    test_manifest_etsy = test_manifest_etsy.sort_values(["source", "image_id"]).reset_index(drop=True)

display(train_manifest_etsy_only.head())
display(train_manifest_etsy_plus_external.head())


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio
0,000541e8-6ea3-4ae0-a79b-10bd3bc3a38d.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000
1,00329fac-20e5-41f7-92df-05b256fdf9e8.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000
2,005af84d-d9ae-4842-a4a2-75fc8f590a94.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000
3,0068b91f-794a-4520-83a4-0f8735d2a229.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,896,1195,RGB,JPEG,0.749791
4,009ad604-af12-4c5d-b6a6-81da359c13a1.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,558,RGB,JPEG,1.835125


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio
0,ai5_3/Ai_generated_dataset/animals/image-10.png,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,ai_generated,ai5_3,train,True,True,512,512,RGB,PNG,1.0
1,ai5_3/Ai_generated_dataset/animals/image-11.png,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,ai_generated,ai5_3,train,True,True,512,512,RGB,PNG,1.0
2,ai5_3/Ai_generated_dataset/animals/image-12.png,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,ai_generated,ai5_3,train,True,True,512,512,RGB,PNG,1.0
3,ai5_3/Ai_generated_dataset/animals/image-13.png,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,ai_generated,ai5_3,train,True,True,512,512,RGB,PNG,1.0
4,ai5_3/Ai_generated_dataset/animals/image-14.png,/home/tbeliere/Projects/DCU/AML_PJ/data2/other...,ai_generated,ai5_3,train,True,True,512,512,RGB,PNG,1.0


## 11. Save manifests and dataset summary

In [79]:
train_manifest_etsy_only_path = MANIFESTS_DIR / "train_manifest_etsy_only.csv"
test_manifest_etsy_path = MANIFESTS_DIR / "test_manifest_etsy.csv"
train_manifest_etsy_plus_external_path = MANIFESTS_DIR / "train_manifest_etsy_plus_external.csv"
external_manifest_all_path = MANIFESTS_DIR / "train_manifest_external_only.csv"
dataset_summary_path = MANIFESTS_DIR / "dataset_summary.csv"
setup_config_snapshot_path = MANIFESTS_DIR / "setup_config_snapshot.json"

train_manifest_etsy_only.to_csv(train_manifest_etsy_only_path, index=False)
test_manifest_etsy.to_csv(test_manifest_etsy_path, index=False)
train_manifest_etsy_plus_external.to_csv(train_manifest_etsy_plus_external_path, index=False)
external_manifest_all.to_csv(external_manifest_all_path, index=False)

dataset_summary = pd.concat([
    summarise_manifest(train_manifest_etsy_only, "train_manifest_etsy_only"),
    summarise_manifest(external_manifest_all, "train_manifest_external_only"),
    summarise_manifest(train_manifest_etsy_plus_external, "train_manifest_etsy_plus_external"),
    summarise_manifest(test_manifest_etsy, "test_manifest_etsy"),
], ignore_index=True)

dataset_summary.to_csv(dataset_summary_path, index=False)

with open(setup_config_snapshot_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print("Saved:")
print(" -", train_manifest_etsy_only_path)
print(" -", external_manifest_all_path)
print(" -", train_manifest_etsy_plus_external_path)
print(" -", test_manifest_etsy_path)
print(" -", dataset_summary_path)
print(" -", setup_config_snapshot_path)

display(dataset_summary)


Saved:
 - /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/train_manifest_etsy_only.csv
 - /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/train_manifest_external_only.csv
 - /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/train_manifest_etsy_plus_external.csv
 - /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/test_manifest_etsy.csv
 - /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/dataset_summary.csv
 - /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/setup_config_snapshot.json


,manifest_name,n_rows,n_missing_files,n_invalid_images,label__authentic,label__ai_generated,source__etsy,source__ai5_3,label__<NA>
0,train_manifest_etsy_only,4800,0,0,2485.0,2315.0,4800.0,NaN,NaN
1,train_manifest_external_only,995,0,0,745.0,250.0,NaN,995.0,NaN
2,train_manifest_etsy_plus_external,5795,0,0,3230.0,2565.0,4800.0,995.0,NaN
3,test_manifest_etsy,2058,0,0,NaN,NaN,2058.0,NaN,2058.0


## 12. Quick inspection cells

In [80]:
print("Etsy train label distribution:")
display(train_manifest_etsy_only["label"].value_counts(dropna=False))

if not external_manifest_all.empty:
    print("\nExternal train label distribution:")
    display(external_manifest_all.groupby(["source", "label"]).size().rename("count").reset_index())

print("\nCombined train label distribution:")
display(train_manifest_etsy_plus_external.groupby(["source", "label"]).size().rename("count").reset_index())


Etsy train label distribution:


label
authentic       2485
ai_generated    2315
Name: count, dtype: int64


External train label distribution:


,source,label,count
0,ai5_3,ai_generated,250
1,ai5_3,authentic,745



Combined train label distribution:


,source,label,count
0,ai5_3,ai_generated,250
1,ai5_3,authentic,745
2,etsy,ai_generated,2315
3,etsy,authentic,2485


In [81]:
print("Any missing files in final manifests?")
checks = {
    "train_manifest_etsy_only_missing": int((~train_manifest_etsy_only["file_exists"]).sum()),
    "train_manifest_etsy_plus_external_missing": int((~train_manifest_etsy_plus_external["file_exists"]).sum()),
    "test_manifest_etsy_missing": int((~test_manifest_etsy["file_exists"]).sum()),
}
checks


Any missing files in final manifests?


{'train_manifest_etsy_only_missing': 0,
 'train_manifest_etsy_plus_external_missing': 0,
 'test_manifest_etsy_missing': 0}

## 13. Next step

Recommended downstream usage:

- `02_preprocess_images.ipynb`
  - input: one of the training manifests from this notebook
  - output: versioned processed-image manifests

- `03_eda.ipynb`
  - input: a selected manifest
  - goal: compare Etsy-only versus Etsy+external and inspect image distributions

- `04_resnet50_kfold.ipynb`
  - input: selected train manifest + test manifest
  - goal: training, OOF probabilities, threshold tuning, submission

### Suggested first experiments
1. `train_manifest_etsy_only.csv` with original images
2. `train_manifest_etsy_plus_external.csv` with original images
3. only after that, test one or two deterministic preprocessing variants
